# 11 pandas basics

A DataFrame is a table: named columns, each with its own type, and a row index.
A Series is a single column.

In [1]:
import pandas as pd

df = pd.read_csv("../data/patients.csv")
df.head()

,patient_id,site,sex,age,weight_kg,height_cm,hiv_status,enrolled
0,P0001,Kampala,M,30,58.1,150.4,Negative,2025-09-20
1,P0002,Mbarara,M,81,61.4,164.4,Negative,2025-03-23
2,P0003,Gulu,F,24,63.5,169.5,Negative,2025-03-22
3,P0004,Gulu,F,8,NaN,153.5,Positive,2025-04-17
4,P0005,Gulu,M,74,73.3,159.3,Negative,2025-03-24


## First look at any new data

In [2]:
print(df.shape)
df.info()

(300, 8)
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  300 non-null    str    
 1   site        300 non-null    str    
 2   sex         300 non-null    str    
 3   age         300 non-null    int64  
 4   weight_kg   288 non-null    float64
 5   height_cm   300 non-null    float64
 6   hiv_status  300 non-null    str    
 7   enrolled    300 non-null    str    
dtypes: float64(2), int64(1), str(5)
memory usage: 18.9 KB


In [3]:
df.describe()

,age,weight_kg,height_cm
count,300.000000,288.000000,300.000000
mean,42.393333,61.994097,160.898667
std,24.831714,13.815749,11.308087
min,1.000000,21.500000,130.900000
25%,20.000000,52.875000,153.450000
50%,42.000000,61.500000,161.200000
75%,65.000000,71.925000,169.125000
max,84.000000,96.000000,191.300000


In [4]:
df.describe(exclude="number")      # the text columns

,patient_id,site,sex,hiv_status,enrolled
count,300,300,300,300,300
unique,300,5,2,3,201
top,P0001,Kampala,F,Negative,2025-09-06
freq,1,97,153,231,4


In [5]:
print(df["site"].value_counts())
print(df["site"].nunique(), df["site"].unique())

site
Kampala    97
Mbarara    68
Gulu       51
Jinja      48
Mbale      36
Name: count, dtype: int64
5 <StringArray>
['Kampala', 'Mbarara', 'Gulu', 'Mbale', 'Jinja']
Length: 5, dtype: str


## Selecting columns

In [6]:
df["age"].head()

0    30
1    81
2    24
3     8
4    74
Name: age, dtype: int64

In [7]:
df[["patient_id", "age", "site"]].head()

,patient_id,age,site
0,P0001,30,Kampala
1,P0002,81,Mbarara
2,P0003,24,Gulu
3,P0004,8,Gulu
4,P0005,74,Gulu


## Selecting rows: loc and iloc

- `loc` selects by label and condition
- `iloc` selects by position

Always use one of these with a single `[]` pair, `df.loc[rows, columns]`.

In [8]:
df.iloc[0:3, 0:4]

,patient_id,site,sex,age
0,P0001,Kampala,M,30
1,P0002,Mbarara,M,81
2,P0003,Gulu,F,24


In [9]:
df.loc[df["age"] > 80, ["patient_id", "age", "site"]]

,patient_id,age,site
1,P0002,81,Mbarara
14,P0015,84,Mbarara
66,P0067,83,Kampala
71,P0072,81,Gulu
72,P0073,81,Mbale
88,P0089,83,Mbarara
125,P0126,84,Kampala
131,P0132,83,Mbarara
155,P0156,83,Kampala
167,P0168,83,Kampala


## Filtering

Combine conditions with `&` (and), `|` (or), `~` (not). Each condition needs brackets.

In [10]:
adults_kla = df.loc[(df["age"] >= 18) & (df["site"] == "Kampala")]
len(adults_kla)

73

In [11]:
df.loc[df["site"].isin(["Gulu", "Mbale"]) & ~(df["hiv_status"] == "Unknown")].head()

,patient_id,site,sex,age,weight_kg,height_cm,hiv_status,enrolled
2,P0003,Gulu,F,24,63.5,169.5,Negative,2025-03-22
3,P0004,Gulu,F,8,NaN,153.5,Positive,2025-04-17
4,P0005,Gulu,M,74,73.3,159.3,Negative,2025-03-24
5,P0006,Mbale,F,18,60.5,180.7,Negative,2025-05-09
13,P0014,Mbale,M,76,52.2,175.3,Negative,2025-01-21


`query()` reads more like a sentence:

In [12]:
df.query("age >= 65 and sex == 'F'").head()

,patient_id,site,sex,age,weight_kg,height_cm,hiv_status,enrolled
18,P0019,Kampala,F,74,57.0,177.6,Negative,2025-10-03
21,P0022,Gulu,F,71,74.7,164.9,Negative,2025-09-06
30,P0031,Kampala,F,73,61.9,175.2,Positive,2025-06-25
40,P0041,Mbarara,F,74,79.0,159.2,Positive,2025-11-21
44,P0045,Kampala,F,67,71.9,137.0,Negative,2025-05-05


## New columns

In [13]:
df["bmi"] = (df["weight_kg"] / (df["height_cm"] / 100) ** 2).round(1)
df["age_group"] = pd.cut(df["age"], bins=[0, 17, 49, 120], labels=["<18", "18-49", "50+"])
df[["age", "age_group", "bmi"]].head()

,age,age_group,bmi
0,30,18-49,25.7
1,81,50+,22.7
2,24,18-49,22.1
3,8,<18,NaN
4,74,50+,28.9


`assign()` does the same and fits in a chain:

In [14]:
(
    df
    .assign(height_m=lambda d: d["height_cm"] / 100)
    .loc[:, ["patient_id", "height_cm", "height_m"]]
    .head(3)
)

,patient_id,height_cm,height_m
0,P0001,150.4,1.504
1,P0002,164.4,1.644
2,P0003,169.5,1.695


## Sorting

In [15]:
df.sort_values(["site", "age"], ascending=[True, False]).head()

,patient_id,site,sex,age,weight_kg,height_cm,hiv_status,enrolled,bmi,age_group
256,P0257,Gulu,F,83,58.0,146.2,Negative,2025-06-23,27.1,50+
71,P0072,Gulu,F,81,60.0,130.9,Negative,2025-04-10,35.0,50+
243,P0244,Gulu,F,81,41.3,153.1,Negative,2025-10-14,17.6,50+
25,P0026,Gulu,M,79,60.3,138.6,Negative,2025-08-31,31.4,50+
239,P0240,Gulu,M,79,78.0,152.5,Negative,2025-03-12,33.5,50+


In [16]:
df.nlargest(3, "weight_kg")[["patient_id", "weight_kg"]]

,patient_id,weight_kg
297,P0298,96.0
37,P0038,94.9
132,P0133,93.1


## Summaries of one column

In [17]:
print(df["age"].mean(), df["age"].median(), df["age"].std())
print(df["weight_kg"].isna().sum(), "missing weights")

42.39333333333333 42.0 24.83171363860445
12 missing weights


## Saving

In [18]:
df.to_csv("patients_with_bmi.csv", index=False)
# df.to_excel("patients.xlsx", index=False)    # needs openpyxl
# df.to_parquet("patients.parquet")             # fast and keeps types, needs pyarrow

from pathlib import Path
Path("patients_with_bmi.csv").unlink()

## Copies and SettingWithCopy

Since pandas 3.0, a filtered DataFrame always behaves like a copy (copy-on-write),
so changing it never changes the original. Assign changes with `.loc` on the
DataFrame you mean to change:

```python
df.loc[df["age"] > 120, "age"] = None     # right
df[df["age"] > 120]["age"] = None         # does nothing to df
```